# Notebook-2

The goal of this notebook is to answer:
- What does "engagement" look like?
- How do we convert behavior into ML-friendly signals?
- What exactly are we predicting?

# Loan Clean Dataset

In [1]:
import pandas as pd
from pathlib import  Path

In [2]:
try:
    root_folder_path=Path(__file__).resolve().parent.parent.parent

except:
    root_folder_path=Path().resolve().parent.parent.parent

print(root_folder_path)

D:\AI-ML\REAL WORLD PROJECTS


In [7]:
base_path = Path(root_folder_path) / "MACHINE LEARNING/Academic-Risk-Engagement-Prediction-System/Dataset"
base_path

WindowsPath('D:/AI-ML/REAL WORLD PROJECTS/MACHINE LEARNING/Academic-Risk-Engagement-Prediction-System/Dataset')

In [10]:
df=pd.read_csv(base_path/'student_master.csv')

In [11]:
list(df.columns)

['code_module',
 'code_presentation',
 'id_student',
 'gender',
 'region',
 'highest_education',
 'imd_band',
 'age_band',
 'num_of_prev_attempts',
 'studied_credits',
 'disability',
 'final_result',
 'total_click',
 'avg_clicks_per_day',
 'max_clicks_day',
 'active_days']

# Handle Missing Engagement Values

In [12]:
engagement_cols=['total_click',
                 'avg_clicks_per_day',
                 'max_clicks_day',
                 'active_days'
                 ]
df[engagement_cols]=df[engagement_cols].fillna(0)

In [13]:
df.head()

,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result,total_click,avg_clicks_per_day,max_clicks_day,active_days
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,Pass,934.0,4.765306,76.0,40.0
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,N,Pass,1435.0,3.337209,23.0,80.0
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,Y,Withdrawn,281.0,3.697368,23.0,12.0
3,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,N,Pass,2158.0,3.254902,22.0,123.0
4,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,N,Pass,1034.0,2.937500,22.0,70.0


# Understanding the target (final_result)

In [14]:
df['final_result'].value_counts()

final_result
Pass           12361
Withdrawn      10156
Fail            7052
Distinction     3024
Name: count, dtype: int64

# Design Decision

For real-world usefullness, we will do : binary risk predict

Pass, Distinction --> 1 (successfull)
Withdrawn, Fail --> 0 (At risk)

In [17]:
df['sucess']=df['final_result'].map({
    'Pass':1,
    'Distinction':1,
    'Withdrawn':0,
    'Fail':0
})

In [18]:
df.head(1)

,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result,total_click,avg_clicks_per_day,max_clicks_day,active_days,sucess
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,Pass,934.0,4.765306,76.0,40.0,1


Successful students --> higher click, more active days

At-risk students --> near zero activity

ML will work in our problem statement.

# Create engagement intensity bands

Instead of raw numers, we will create better/human interpretable categories using feature bucketing.


In [19]:
df.head(1)

,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result,total_click,avg_clicks_per_day,max_clicks_day,active_days,sucess
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,Pass,934.0,4.765306,76.0,40.0,1


In [21]:
df['engagement_level']=pd.qcut(
    df['total_click'],
    q=3,
    labels=['Low','Medium','High']
)

In [23]:
df.head(2)

,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result,total_click,avg_clicks_per_day,max_clicks_day,active_days,sucess,engagement_level
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,Pass,934.0,4.765306,76.0,40.0,1,Medium
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,N,Pass,1435.0,3.337209,23.0,80.0,1,High


# Absence of engagement

In [24]:
df['no_engagement_flag']=(df['active_days']==0).astype(int)

# Save Dataset For ML

In [25]:
output_path = Path(root_folder_path) / "MACHINE LEARNING/Academic-Risk-Engagement-Prediction-System/Dataset" / "student_master_v2.csv"
df.to_csv(output_path, index=False)
